# File

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
df = pd.read_csv("TORUN.csv") # ETO YUNG GINAMIT KONG CSV YA CHINECK Q NA
X = df.drop('HIV_Status', axis=1)
y = df['HIV_Status']

# Splitting
X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, test_size=0.2, random_state=42
)

In [4]:
df.head(10)

,Sex,Age_Group,Transmission,Healthcare_Access_Friction,Civil_Status,OFW_Status,Chemsex_Engagement,Alcohol_Sex_Risk,PrEP_Awareness,Transactional_Sex,STI_BBV_CoInfection_Any,HIV_Status
0,Female,<15,Male to Female Sex,2,Single,No,No,No,No,No TS,No,Reactive
1,Male,15-24,Male to Female Sex,2,Single,No,No,No,No,No TS,No,Reactive
2,Male,15-24,Male to Male Sex,2,Single,No,No,No,Yes,No TS,Yes,Reactive
3,Male,15-24,Male to Male Sex,2,Single,No,No,No,No,No TS,Yes,Reactive
4,Male,15-24,Male to Male Sex,2,Single,No,Yes,No,Yes,No TS,No,Reactive
5,Male,15-24,Male to Male/Female Sex,2,Single,No,No,Yes,Yes,Both,No,Reactive
6,Male,25-34,Male to Female Sex,2,Common-Law,No,No,No,No,No TS,No,Reactive
7,Male,25-34,Male to Male Sex,2,Single,No,No,No,Yes,No TS,No,Reactive
8,Male,25-34,Male to Male Sex,2,Single,No,No,No,No,No TS,No,Reactive
9,Male,25-34,Male to Male Sex,2,Single,No,No,Yes,Yes,Paid for sex,Yes,Reactive


# Preprocessing

In [5]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# list num and cat
numeric_features = []
categorical_features = ['Sex','Age_Group','Transmission',
                        'Healthcare_Access_Friction','Civil_Status','OFW_Status','Chemsex_Engagement',
                        'Alcohol_Sex_Risk','PrEP_Awareness','Transactional_Sex','STI_BBV_CoInfection_Any']

# 2. proprocessor
preprocessor = ColumnTransformer(
    transformers=[
        # Apply StandardScaler to numeric columns
        ('num', StandardScaler(), numeric_features),

        # Apply OneHotEncoder to categorical columns
        # handle_unknown='ignore' ensures the code won't crash if X_test has a category not seen in X_train
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ],
    remainder='passthrough' # Keep any other columns not listed above (or use 'drop')
)

# 3. Fit and Transform X_train
# CRITICAL: We fit ONLY on X_train to learn the mean/std and categories.
# Then we transform both train and test using those learned values.
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# convert into df
X# Convert into DataFrame AND preserve the original index
X_train_processed = pd.DataFrame(
    X_train_processed,
    columns=preprocessor.get_feature_names_out(),
    index=X_train.index
)

X_test_processed = pd.DataFrame(
    X_test_processed,
    columns=preprocessor.get_feature_names_out(),
    index=X_test.index
)

# Dealing with y columns
mapping = {'Non-Reactive': 0, 'Reactive': 1}

# y_train and y_test already keep their indices, but it's good practice
# to ensure it's a Series or 1D array for Scikit-Learn classifiers
y_train_processed = y_train.map(mapping)
y_test_processed = y_test.map(mapping)

 # MODEL

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score, average_precision_score

# 1. Initialize Base Logistic Regression
# max_iter=1000 ensures it has enough time to find the mathematical optimal point
lr_base = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=42,
    solver='liblinear' # 'liblinear' is great for smaller datasets and supports l1/l2 penalties
)

# 2. Define the Hyperparameter Grid
param_grid_lr = {
    'C': [0.01, 0.1, 1, 10, 100], # Inverse of regularization strength
    'penalty': ['l1', 'l2']       # L1 can actually drop useless columns completely!
}

# 3. Setup Grid Search optimized for the minority class (AUPRC)
grid_search_lr = GridSearchCV(
    estimator=lr_base,
    param_grid=param_grid_lr,
    scoring='average_precision',
    cv=5,
    verbose=1,
    n_jobs=-1
)

# 4. Train the Model!
print("Training Logistic Regression...")
grid_search_lr.fit(X_train_processed, y_train_processed)

best_lr_model = grid_search_lr.best_estimator_

# 5. Make Predictions
y_pred_lr = best_lr_model.predict(X_test_processed)
y_probs_lr = best_lr_model.predict_proba(X_test_processed)[:, 1]

# 6. Evaluate
print("\n--- Logistic Regression Final Report ---")
print(f"Best Parameters: {grid_search_lr.best_params_}")
print(classification_report(y_test_processed, y_pred_lr, target_names=['Non Reactive', 'Reactive']))
print(f"LR F1-Score: {f1_score(y_test_processed, y_pred_lr):.4f}")
print(f"LR AUPRC: {average_precision_score(y_test_processed, y_probs_lr):.4f}")

Training Logistic Regression...
Fitting 5 folds for each of 10 candidates, totalling 50 fits


/opt/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.che

# Threshold Moving 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (
    f1_score, 
    precision_score, 
    recall_score,
    precision_recall_curve,       
    average_precision_score       
)


# 1. Get the probabilities for the 'Reactive' class (index 1)
lr_probs = best_lr_model.predict_proba(X_test_processed)[:, 1]

# 2. Test every threshold from 0.01 to 0.99
thresholds = np.linspace(0.01, 0.99, 99)
f1_scores = []
precisions = []
recalls = []

for t in thresholds:
    # Classify as 1 if probability >= t
    preds = (lr_probs >= t).astype(int)
    f1_scores.append(f1_score(y_test_processed, preds))
    precisions.append(precision_score(y_test_processed, preds, zero_division=0))
    recalls.append(recall_score(y_test_processed, preds))

# 3. Find the best threshold
best_idx = np.argmax(f1_scores)
best_threshold_lr = thresholds[best_idx]
best_f1_lr = f1_scores[best_idx]


In [ ]:
# ---Calculate PR Curve data! ---
precision_lr, recall_lr, pr_thresholds_lr = precision_recall_curve(y_test_processed, lr_probs)
auprc_lr = average_precision_score(y_test_processed, lr_probs)

# 4. Create the Figure
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))

# --- PLOT 1: Threshold vs Metrics ---
# (Fixed variable names to match the lists generated in Step 2)
ax1.plot(thresholds, f1_scores, label='F1-Score', color='#39C5BB', linewidth=4)
ax1.plot(thresholds, precisions, label='Precision', color='#FFBACC', linestyle='--', alpha=0.8)
ax1.plot(thresholds, recalls, label='Recall', color='#FFCC11', linestyle='--', alpha=0.8)

# Highlight the Best Threshold
ax1.axvline(best_threshold_lr, color='#DD4444', linestyle=':', label=f'Best Threshold: {best_threshold_lr:.2f}')
ax1.fill_between(thresholds, 0, f1_scores, color='#39C5BB', alpha=0.1)

ax1.set_title(f"Logistic Regression Threshold Optimization\nMax F1: {best_f1_lr:.4f}", fontsize=14)
ax1.set_xlabel("Probability Threshold")
ax1.set_ylabel("Score")
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

# --- PLOT 2: Precision-Recall Curve ---
ax2.plot(recall_lr, precision_lr, color='#39C5BB', linewidth=4, label=f'PR Curve (AUPRC = {auprc_lr:.4f})')
ax2.fill_between(recall_lr, 0, precision_lr, color='#39C5BB', alpha=0.2)

# Mark the specific point on the curve where our "Best Threshold" sits
idx_lr = np.argmin(np.abs(pr_thresholds_lr - best_threshold_lr))
ax2.scatter(recall_lr[idx_lr], precision_lr[idx_lr], color='red', s=100, zorder=5, label='Chosen Operating Point')

ax2.set_title("Logistic Regression Precision-Recall Curve\n(The 'Imbalance' Gold Standard)", fontsize=14)
ax2.set_xlabel("Recall (Ability to find Reactives)")
ax2.set_ylabel("Precision (Ability to be correct)")
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"--- Logistic Regression Final Audit ---")
print(f"Old F1 (50%): {f1_score(y_test_processed, best_lr_model.predict(X_test_processed)):.4f}")
print(f"New F1 ({best_threshold_lr*100:.0f}%): {best_f1_lr:.4f}")

# SHAP

In [ ]:
import shap

# 1. Initialize the Linear Explainer for Logistic Regression
# Linear models usually need the training data passed in to establish a "baseline"
explainer_lr = shap.LinearExplainer(best_lr_model, X_train_processed)
shap_vals_lr = explainer_lr.shap_values(X_test_processed)

# Unlike Trees (which return [Class 0, Class 1]), LinearExplainer for binary LR
# usually just returns a single array for the positive class.
# We add a quick safety check just in case your specific version returns a list!
if isinstance(shap_vals_lr, list):
    shap_raw_lr = shap_vals_lr[1]
else:
    shap_raw_lr = shap_vals_lr

# Put them in a DataFrame for easy grouping
shap_df_lr = pd.DataFrame(shap_raw_lr, columns=X_test_processed.columns)
grouped_shap_df_lr = shap_df_lr.copy()

# 2. Define the exact prefixes of your One-Hot Encoded columns
categorical_prefixes = ['cat__Sex','cat__Age_Group','cat__Transmission',
                        'cat__Healthcare_Access_Friction','cat__Civil_Status','cat__OFW_Status','cat__Chemsex_Engagement',
                        'cat__Alcohol_Sex_Risk','cat__PrEP_Awareness','cat__Transactional_Sex','cat__STI_BBV_CoInfection_Any']

# 3. Stitch the dummy columns back together!
for prefix in categorical_prefixes:
    # Find all columns that start with this prefix
    dummy_cols = [col for col in X_test_processed.columns if col.startswith(f"{prefix}_")]

    if len(dummy_cols) > 0:
        # Sum the dummy parts to get the total category importance
        grouped_shap_df_lr[prefix] = shap_df_lr[dummy_cols].sum(axis=1)
        # Drop the diluted dummy columns
        grouped_shap_df_lr.drop(columns=dummy_cols, inplace=True)

In [ ]:
# 4. Plot the Global Feature Importance (Miku Teal!)
plt.figure(figsize=(12, 8))
plt.title("Logistic Regression: True Feature Importance (Grouped)", fontsize=14)

shap.summary_plot(
    grouped_shap_df_lr.values,
    feature_names=grouped_shap_df_lr.columns,
    plot_type="bar",
    color="#39C5BB",  #Turquoise
    show=False
)

plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## TOP 10 INDIVIDUAL FEATURES 

In [ ]:
# 1. Calculate SHAP values
explainer_lr = shap.LinearExplainer(best_lr_model, X_train_processed)
shap_vals_lr = explainer_lr.shap_values(X_test_processed)

# 2.(Just in case it returns a list!)
if isinstance(shap_vals_lr, list):
    shap_raw_lr = shap_vals_lr[1]
else:
    shap_raw_lr = shap_vals_lr

# 3. The Plot
plt.figure(figsize=(10, 6))

shap.summary_plot(
    shap_raw_lr,                           
    features=X_test_processed,
    feature_names=X_test_processed.columns,
    plot_type="bar",                        
    color="#39C5BB",                         # Miku Teal!
    max_display=10,                          # Top 10 only
    show=False
)

# Optional: Adding a subtle grid
plt.grid(axis='x', color='gray', alpha=0.2, linestyle=':')
plt.tight_layout()
plt.show()

## Beeswarm

In [ ]:
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt

# 1. Set up the Miku gradient
miku_cmap = mcolors.LinearSegmentedColormap.from_list("miku_gradient", ["#D1F2F0", "#39C5BB"])

# 2. Create the canvas 
plt.figure(figsize=(10, 6))

# 3. Draw the plot
shap.summary_plot(
    shap_vals_lr,                            # (Or shap_raw_lr if you used the safety check!)
    features=X_test_processed,
    feature_names=X_test_processed.columns,
    plot_type="dot",                         # Beeswarm style
    cmap=miku_cmap,
    max_display=10,                          # Top 10 only
    show=False
)

# Optional: Adding a subtle grid
plt.grid(axis='x', color='gray', alpha=0.2, linestyle=':')
plt.tight_layout()
plt.show()

# SAVING THE MODEL (RUN MO TO YA IF YOU THINK MAGANDA RESULT PARA MASAVE)

In [ ]:
model_filename_lr = 'best_lr_model.joblib'
joblib.dump(best_lr_model, model_filename_lr)

print(f"\nModel successfully saved to {model_filename_lr}")